In [ ]:
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import visualization.visualization as vis
import dl_embedding.mae as mae
import metrics.embedding_eval as ev
from cuml.cluster import HDBSCAN
from cuml.manifold import UMAP

import torch
import numpy as np
from matplotlib import pyplot as plt


In [2]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2",
                                          run_id="1_00",
                                          dataset_name="cutout_dataset.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io"
                                          )

source.print_available_channels()

23 available channels:
['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl','SIarea','XC','YC']


In [3]:
#data_channels = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl']
#data_channels_high_res = ['Salt','Theta','U','V','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']
data_channels_no_cor = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']

data_channels_engineered = ['gradb2','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']
dataset = cutouts_dataset.CutoutDataset.from_source(data_channels=data_channels_engineered, source=source, subset=False,
                                                    subsample_per_chunk=64, num_sample_chunks=1, n_workers=4)

/home/jovyan/conda_envs/main_cuml/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45345 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:34739' processes=4 threads=32, memory=128.00 GiB>
nrp link url : https://jupyterhub-west.nrp-nautilus.io/hub/user-redirect/proxy/45345/status
dropped 0 ice, 0 NaN; kept 3250 / 3250
features ['gradb2', 'gradrho2', 'turner_angle', 'strain_n', 'strain_s', 'strain_mag', 'divergence', 'relative_vorticity', 'oceQnet', 'ekman_pumping', 'wind_stress_curl'] | coords ['XC', 'YC']


2026-07-27 23:55:30,103 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 23:55:30,105 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 23:55:30,105 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-07-27 23:55:30,106 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


In [ ]:
# cutouts as MAE input (64x64 -> 8x8 token grid at patch_size 8);
# hold out 20% of cutouts to monitor validation reconstruction loss per epoch
patch_size = 8
images = dataset.preprocess_for_training()          # (N, C, 64, 64): log-grads + z-score
rng = np.random.default_rng(0)
perm = rng.permutation(len(images))
n_val = max(1, int(0.2 * len(images)))
images_val, images_train = images[perm[:n_val]], images[perm[n_val:]]
print(images.shape, "| train", len(images_train), "| val", len(images_val))


In [ ]:
# train the MAE on our cutouts (per-epoch train/val reconstruction loss)
embedder = mae.MAEEmbedder(patch_size=patch_size, mask_ratio=0.75)
embedder.fit(images_train, val_images=images_val, epochs=50)


In [ ]:
# encode ALL cutouts -> per-patch embeddings for clustering + evaluation
embeddings = embedder.embed(images)                 # (N * (64/8)^2, embed_dim)
print(embeddings.shape)


In [ ]:
# Cluster the learned embeddings with cuML HDBSCAN
clusterer = HDBSCAN(min_cluster_size=5, min_samples=None)
clusters = np.asarray(clusterer.fit_predict(embeddings))
print("clusters", int(clusters.max()) + 1,
      "| noise", int((clusters == -1).sum()), "/", clusters.size)


In [ ]:
# 3D view of the clustering (cuML UMAP to 3D for display only)
emb3 = np.asarray(UMAP(n_components=3, n_neighbors=15, min_dist=0.0).fit_transform(embeddings))
vis.vis_dim_redux(emb3, labels=clusters, dims=3, alpha=0.5)


In [ ]:
np.unique(clusters)


In [ ]:
vis.plot_global_cluster_maps(dataset, clusters, patch_size=patch_size, alpha=0.1,
                             point_size=10.0, extent=None, coastlines=True,
                             panel_size=8, drop_noise=False, save_dir=None)


In [ ]:
vis.make_image_from_patches(dataset, clusters, patch_size=patch_size, number_rows=100)


In [ ]:
# patch semantic-similarity validation: descriptors
desc, lon, lat, names = ev.patch_descriptors(dataset, patch_size)
print(desc.shape)


In [ ]:
# do nearest neighbors share physical properties / geography?
ev.neighbor_consistency(embeddings, desc, names, k=10)
ev.neighbor_geographic(embeddings, lon, lat, k=10)


In [ ]:
# eyeball: query patches and their nearest neighbors (colored by strain_mag)
ev.plot_neighbor_examples(dataset, embeddings, patch_size,
                          query_idx=[0, 100, 500, 1000], k=8, channel="strain_mag")
